# E7: Feature Selection Funnel — L3 (Distribution Stability / PSI)

**Spec sections**: 6.4 (all), 6.8, D.4, 11.2 (row 1)  
**Population P1**: `sample_status` ∈ `config.psi_statuses` (GOOD, BAD, INDETERMINATE, IMMATURE, EXCLUDED_PERF). No OOT.  
**Input**: Features with `L2_pass = Y`; L1 binning objects (fixed).

In [0]:
%pip install "optbinning" "numpy<2" "pyarrow==14.0.1" --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, hashlib, shutil, tempfile, json, time, pickle, warnings
from datetime import datetime

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

import numpy as np
import pandas as pd
import common

warnings.filterwarnings("ignore", category=FutureWarning)

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
ARTIFACTS = common.artifacts_path(cfg)
REPORTS   = os.path.join(ARTIFACTS, "reports")
BINNING_DIR = os.path.join(ARTIFACTS, "binning", "L1")
os.makedirs(REPORTS, exist_ok=True)

cat = cfg["databricks"]["catalog"]
sch = cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.acard_master"
FLOG_TBL   = f"{cat}.{sch}.acard_feature_log"

L3 = cfg["funnel"]["L3"]
PSI_GREEN     = L3["psi_green"]
PSI_RED       = L3["psi_red"]
EPS           = L3["eps"]
MISSING_JUMP  = L3["missing_jump"]

target_col = cfg["columns"]["target"]
status_col = cfg["columns"]["status"]
PSI_STATUSES = cfg["psi_statuses"]
SEED = cfg["random_seed"]

# Funnel config hash check
funnel_config_hash = hashlib.sha256(
    json.dumps(cfg["funnel"], sort_keys=True).encode()
).hexdigest()

run_summary_path = os.path.join(REPORTS, "run_summaries", "s03_L0.json")
assert os.path.exists(run_summary_path), f"L0 run summary not found: {run_summary_path}"
with open(run_summary_path) as f:
    l0_summary = json.load(f)
l0_funnel_hash = l0_summary["funnel_config_hash"]

tc_path = os.path.join(PROJECT_ROOT, "config", "threshold_changes.csv")
if funnel_config_hash != l0_funnel_hash:
    if os.path.exists(tc_path):
        tc = pd.read_csv(tc_path)
        print(f"\u26a0 Funnel config hash changed but threshold_changes.csv has {len(tc)} row(s) — proceeding.")
    else:
        raise ValueError(
            f"Funnel config hash mismatch!\n"
            f"  L0 frozen : {l0_funnel_hash[:12]}\n"
            f"  Current   : {funnel_config_hash[:12]}\n"
            f"Add a row to config/threshold_changes.csv explaining the change."
        )
else:
    print(f"\u2705 Funnel config hash matches L0: {funnel_config_hash[:12]}")

_TMP = tempfile.mkdtemp(prefix="e7_")
_t0 = time.time()

print(f"Config hash       : {common.config_hash(cfg)[:12]}")
print(f"PSI statuses      : {PSI_STATUSES}")
print(f"L3 params         : psi_green={PSI_GREEN}, psi_red={PSI_RED}, eps={EPS}, missing_jump={MISSING_JUMP}")

In [0]:
# ── Load master table: P1 population (spec 6.4.1) ─────────────────
master = spark.table(MASTER_TBL).toPandas()

# P1: sample_status in psi_statuses, EXCLUDED removed
# Baseline and comparison use EXACTLY the same filter (spec 6.4.1)
p1_mask = master[status_col].isin(PSI_STATUSES)

dev_p1  = master[(master["window"] == "DEV") & p1_mask].copy()
fwd_p1  = master[(master["window"] == "FWD") & p1_mask].copy()
stress_p1 = master[(master["window"] == "STRESS") & p1_mask].copy()

# OOT is NEVER read for L3
oot_count = len(master[(master["window"] == "OOT") & p1_mask])
print(f"OOT P1 rows available but NOT loaded: {oot_count}")

print(f"DEV P1 rows    : {len(dev_p1):,}")
print(f"FWD P1 rows    : {len(fwd_p1):,}")
print(f"STRESS P1 rows : {len(stress_p1):,}")

# Also load DEV GOOD+BAD for the POLICY head-tail check (step 5)
dev_gb = master[
    (master["window"] == "DEV") & (master[status_col].isin(cfg["labelled_statuses"]))
].copy()
y_gb = (dev_gb[status_col] == "BAD").astype(int)
print(f"DEV GOOD+BAD   : {len(dev_gb):,}  Bads: {y_gb.sum()}")

# Feature log: L2 survivors
flog = spark.table(FLOG_TBL).toPandas()
l2_pass_feats = flog[flog["L2_pass"] == "Y"]["feature"].tolist()
l2_pass_feats = [f for f in l2_pass_feats if f in dev_p1.columns]
print(f"L2 survivors   : {len(l2_pass_feats)}")

# Load L1 binning objects
binning_objects = {}
for feat in l2_pass_feats:
    pkl_path = os.path.join(BINNING_DIR, f"{feat}.pkl")
    assert os.path.exists(pkl_path), f"Missing L1 binning object: {pkl_path}"
    with open(pkl_path, "rb") as f:
        binning_objects[feat] = pickle.load(f)
print(f"Loaded {len(binning_objects)} L1 binning objects")

In [0]:
# ── Convert features to L1 bin indices (spec 6.4.2) ──────────────

def safe_transform(ob, X):
    """Transform with numeric coercion fallback for mixed-type columns."""
    try:
        return pd.Series(ob.transform(X, metric="bins")).astype(str)
    except TypeError:
        return pd.Series(ob.transform(pd.to_numeric(X, errors="coerce"), metric="bins")).astype(str)

def build_bin_df(data, features, binning_objects):
    """Build a DataFrame of bin indices for all features."""
    result = {}
    for feat in features:
        result[feat] = safe_transform(binning_objects[feat], data[feat]).values
    return pd.DataFrame(result, index=data.index)

print("Building bin-index DataFrames...")
dev_bins = build_bin_df(dev_p1, l2_pass_feats, binning_objects)
dev_q = dev_p1["app_quarter"].astype(str)

fwd_bins = build_bin_df(fwd_p1, l2_pass_feats, binning_objects) if len(fwd_p1) > 0 else None
stress_bins = build_bin_df(stress_p1, l2_pass_feats, binning_objects) if len(stress_p1) > 0 else None

# Also build bins for DEV GOOD+BAD (for POLICY head-tail check)
dev_gb_bins = build_bin_df(dev_gb, l2_pass_feats, binning_objects)

print(f"DEV bins       : {dev_bins.shape}")
print(f"FWD bins       : {fwd_bins.shape if fwd_bins is not None else 'None (no FWD data)'}")
print(f"STRESS bins    : {stress_bins.shape if stress_bins is not None else 'None (no STRESS data)'}")
print(f"DEV G+B bins   : {dev_gb_bins.shape}")

In [0]:
# ── PSI calculations (spec 6.4.3) ───────────────────────────────

# Use common.l3_psi_table for ① primary, ② adjacent, ③ FWD
print("Computing PSI table (primary + adjacent + FWD)...")
psi_df = common.l3_psi_table(dev_bins, dev_q, fwd_bins, l2_pass_feats, cfg)

if fwd_bins is None:
    print("\u26a0 FWD data not available — psi_fwd set to null; status determined from psi_max_dev only.")

print(f"\nPSI status distribution:")
print(psi_df["psi_status"].value_counts().to_string())

# Also build per-quarter PSI matrix for the report
quarters = sorted(dev_q.unique())
psi_q_matrix = []
for feat in l2_pass_feats:
    full = dev_bins[feat]
    row = {"feature": feat}
    for q in quarters:
        row[str(q)] = common.psi_from_bins(full, dev_bins.loc[dev_q == q, feat], EPS)
    psi_q_matrix.append(row)
psi_q_df = pd.DataFrame(psi_q_matrix)

# Adjacent-quarter PSI matrix
psi_adj_matrix = []
for feat in l2_pass_feats:
    row = {"feature": feat}
    for a, b in zip(quarters[:-1], quarters[1:]):
        row[f"{a}->{b}"] = common.psi_from_bins(
            dev_bins.loc[dev_q == a, feat],
            dev_bins.loc[dev_q == b, feat], EPS
        )
    psi_adj_matrix.append(row)
psi_adj_df = pd.DataFrame(psi_adj_matrix)

print(f"Quarterly PSI matrix: {psi_q_df.shape}")
print(f"Adjacent PSI matrix : {psi_adj_df.shape}")

In [0]:
# ── STRESS monthly PSI (spec 6.4.3 ④, 11.2 row 1) ─────────────
# STRESS PSI is diagnosis only, NEVER used to drop features

stress_cols = []
if stress_bins is not None and len(stress_p1) > 0:
    stress_months = sorted(stress_p1["app_month"].astype(str).unique())
    print(f"STRESS months: {stress_months}")
    stress_psi_rows = []
    for feat in l2_pass_feats:
        full = dev_bins[feat]
        row = {"feature": feat}
        for mi, m in enumerate(stress_months, 1):
            mask = stress_p1["app_month"].astype(str) == m
            if mask.sum() > 0:
                row[f"psi_stress_m{mi}"] = common.psi_from_bins(full, stress_bins.loc[mask, feat], EPS)
            else:
                row[f"psi_stress_m{mi}"] = np.nan
        stress_psi_rows.append(row)
    stress_psi_df = pd.DataFrame(stress_psi_rows)
    stress_cols = [c for c in stress_psi_df.columns if c.startswith("psi_stress_")]
    print(f"STRESS PSI columns: {stress_cols}")
else:
    stress_psi_df = pd.DataFrame({"feature": l2_pass_feats})
    # Add 7 null stress columns per spec
    for i in range(1, 8):
        stress_psi_df[f"psi_stress_m{i}"] = np.nan
    stress_cols = [f"psi_stress_m{i}" for i in range(1, 8)]
    print("\u26a0 No STRESS data available — psi_stress_m1..m7 set to null.")

print(f"STRESS PSI rows: {len(stress_psi_df)}")

In [0]:
# ── Missing-rate jump check (spec 6.4.6) ────────────────────

missing_rate_rows = []
for feat in l2_pass_feats:
    row = {"feature": feat}
    full_mr = dev_p1[feat].isna().mean()
    row["full_dev_mr"] = full_mr
    max_diff_from_full = 0.0
    max_diff_from_prev = 0.0
    prev_mr = None
    for q in quarters:
        q_data = dev_p1.loc[dev_q == q, feat]
        q_mr = q_data.isna().mean()
        row[str(q)] = q_mr
        diff_full = abs(q_mr - full_mr)
        max_diff_from_full = max(max_diff_from_full, diff_full)
        if prev_mr is not None:
            max_diff_from_prev = max(max_diff_from_prev, abs(q_mr - prev_mr))
        prev_mr = q_mr
    row["max_diff_from_full"] = max_diff_from_full
    row["max_diff_from_prev"] = max_diff_from_prev
    row["missing_jump"] = "Y" if max(max_diff_from_full, max_diff_from_prev) > MISSING_JUMP else "N"
    missing_rate_rows.append(row)

missing_rate_df = pd.DataFrame(missing_rate_rows)
n_jump = (missing_rate_df["missing_jump"] == "Y").sum()
print(f"Missing-rate jump: {n_jump} features flagged (threshold={MISSING_JUMP})")

In [0]:
# ── Red-feature decisions (spec 6.4.4, 6.4.5) ────────────────

red_feats = psi_df[psi_df["psi_status"] == "red"]["feature"].tolist()
print(f"Red features: {len(red_feats)}")

# Load psi_decisions.csv if it exists
psi_dec_path = os.path.join(PROJECT_ROOT, "config", "psi_decisions.csv")
if os.path.exists(psi_dec_path):
    psi_dec = pd.read_csv(psi_dec_path)
    print(f"psi_decisions.csv loaded: {len(psi_dec)} rows")
else:
    psi_dec = pd.DataFrame(columns=["feature", "cause_type", "evidence",
                                     "policy_change_date", "decision", "reviewer", "date"])
    print("psi_decisions.csv not found — all red features treated as UNKNOWN -> DROP")

# For each red feature, determine cause_type and decision
red_decisions = []
for feat in red_feats:
    match = psi_dec[psi_dec["feature"] == feat]
    if len(match) == 0:
        # No row -> UNKNOWN, DROP (spec 6.4.5)
        red_decisions.append({
            "feature": feat, "cause_type": "UNKNOWN", "decision": "DROP",
            "evidence": "", "policy_change_date": pd.NaT, "source": "no_row"
        })
    else:
        row = match.iloc[0]
        cause_type = str(row.get("cause_type", "UNKNOWN")).strip().upper()
        decision = str(row.get("decision", "DROP")).strip().upper()
        evidence = str(row.get("evidence", "")).strip()
        pcd = row.get("policy_change_date", pd.NaT)

        # Validate KEEP: only POLICY cause_type with non-empty evidence and policy_change_date
        if decision == "KEEP":
            if cause_type != "POLICY":
                raise ValueError(
                    f"psi_decisions.csv: KEEP for '{feat}' requires cause_type=POLICY, got '{cause_type}'"
                )
            if not evidence:
                raise ValueError(
                    f"psi_decisions.csv: KEEP for '{feat}' requires non-empty evidence"
                )
            if pd.isna(pcd) or str(pcd).strip() == "":
                raise ValueError(
                    f"psi_decisions.csv: KEEP for '{feat}' requires policy_change_date"
                )
        red_decisions.append({
            "feature": feat, "cause_type": cause_type, "decision": decision,
            "evidence": evidence, "policy_change_date": pcd, "source": "csv"
        })

red_dec_df = pd.DataFrame(red_decisions) if red_decisions else pd.DataFrame(
    columns=["feature", "cause_type", "decision", "evidence", "policy_change_date", "source"]
)
print(f"\nRed-feature decisions:")
if len(red_dec_df) > 0:
    print(red_dec_df[["feature", "cause_type", "decision", "source"]].to_string(index=False))
else:
    print("  (no red features)")

In [0]:
# ── POLICY head-tail check (spec 6.4.5) ─────────────────────
# For POLICY + KEEP: check head-tail on DEV GOOD+BAD before vs after policy_change_date
# A reversal -> drop PSI_POLICY_REVERSAL regardless of the decision

policy_keeps = red_dec_df[
    (red_dec_df["decision"] == "KEEP") & (red_dec_df["cause_type"] == "POLICY")
].copy() if len(red_dec_df) > 0 else pd.DataFrame()

policy_reversals = []
for _, pk in policy_keeps.iterrows():
    feat = pk["feature"]
    pcd = pd.Timestamp(pk["policy_change_date"])
    ob = binning_objects[feat]

    app_dates = pd.to_datetime(dev_gb[cfg["columns"]["decision_time"]])
    before_mask = app_dates < pcd
    after_mask = app_dates >= pcd

    if before_mask.sum() < 50 or after_mask.sum() < 50:
        print(f"  {feat}: too few records before/after {pcd.date()} — skipping head-tail check")
        continue

    # Get bin stats before and after
    bins_all = safe_transform(ob, dev_gb[feat])
    y_all = y_gb
    bins_before = bins_all[before_mask]
    bins_after = bins_all[after_mask]
    y_before = y_all[before_mask]
    y_after = y_all[after_mask]

    # Full-DEV non-special bins: highest and lowest BR
    df_full = pd.DataFrame({"bin": bins_all.values, "y": y_all.values})
    full_stats = df_full.groupby("bin").agg(count=("y", "size"), bads=("y", "sum")).reset_index()
    full_stats["br"] = full_stats["bads"] / full_stats["count"]
    full_stats["is_special"] = full_stats["bin"].apply(
        lambda b: str(b).lower() in ["missing", "special"] or str(b).startswith("Special")
    )
    regular = full_stats[~full_stats["is_special"]]
    if len(regular) < 2:
        continue
    head_bin = regular.loc[regular["br"].idxmax(), "bin"]
    tail_bin = regular.loc[regular["br"].idxmin(), "bin"]

    # Check head-tail flip in the "after" period
    df_after = pd.DataFrame({"bin": bins_after.values, "y": y_after.values})
    after_stats = df_after.groupby("bin").agg(count=("y", "size"), bads=("y", "sum")).reset_index()
    after_stats["br"] = after_stats["bads"] / after_stats["count"]

    h_row = after_stats[after_stats["bin"] == head_bin]
    l_row = after_stats[after_stats["bin"] == tail_bin]
    if len(h_row) > 0 and len(l_row) > 0:
        h_br = float(h_row["br"].values[0])
        l_br = float(l_row["br"].values[0])
        if h_br <= l_br:
            policy_reversals.append(feat)
            print(f"  {feat}: HEAD-TAIL REVERSAL after {pcd.date()} (H_BR={h_br:.4f} <= L_BR={l_br:.4f})")
        else:
            print(f"  {feat}: no reversal after {pcd.date()} (H_BR={h_br:.4f} > L_BR={l_br:.4f})")

# Override decision for policy reversals
if len(red_dec_df) > 0:
    for feat in policy_reversals:
        red_dec_df.loc[red_dec_df["feature"] == feat, "decision"] = "DROP"
        red_dec_df.loc[red_dec_df["feature"] == feat, "cause_type"] = "PSI_POLICY_REVERSAL"

if len(policy_keeps) == 0:
    print("No POLICY+KEEP features to check.")
print(f"Policy reversals: {len(policy_reversals)}")

In [0]:
# ── Assign L3_pass (spec 6.4.4) ───────────────────────────────

# Start from psi_df and merge in other columns
l3_df = psi_df.copy()

# Merge missing_jump
l3_df = l3_df.merge(missing_rate_df[["feature", "missing_jump"]], on="feature", how="left")

# Merge red-feature decisions
if len(red_dec_df) > 0:
    l3_df = l3_df.merge(
        red_dec_df[["feature", "cause_type", "decision"]].rename(
            columns={"decision": "psi_decision"}
        ),
        on="feature", how="left"
    )
else:
    l3_df["cause_type"] = pd.NA
    l3_df["psi_decision"] = pd.NA

# Merge STRESS PSI
l3_df = l3_df.merge(stress_psi_df, on="feature", how="left")

# Assign L3_pass
# green/yellow -> Y (yellow is a flag, not a drop)
# red with decision DROP -> N
# red with decision KEEP -> Y
l3_df["L3_pass"] = "Y"
l3_df["L3_reason"] = pd.NA

for idx in l3_df.index:
    status = l3_df.loc[idx, "psi_status"]
    if status == "red":
        decision = l3_df.loc[idx, "psi_decision"]
        if pd.isna(decision) or str(decision).upper() == "DROP":
            l3_df.loc[idx, "L3_pass"] = "N"
            cause = l3_df.loc[idx, "cause_type"]
            l3_df.loc[idx, "L3_reason"] = f"PSI_RED_{cause}" if pd.notna(cause) else "PSI_RED_UNKNOWN"

n_pass = (l3_df["L3_pass"] == "Y").sum()
n_drop = (l3_df["L3_pass"] == "N").sum()
n_green = (l3_df["psi_status"] == "green").sum()
n_yellow = (l3_df["psi_status"] == "yellow").sum()
n_red = (l3_df["psi_status"] == "red").sum()

print(f"L3 results: {n_pass} pass, {n_drop} dropped")
print(f"  green={n_green}, yellow={n_yellow}, red={n_red}")
if n_drop > 0:
    print(f"\nDrop reasons:")
    print(l3_df[l3_df["L3_pass"] == "N"]["L3_reason"].value_counts().to_string())

In [0]:
# ── Update feature log ───────────────────────────────────────────────────

l3_merge_cols = ["feature", "psi_max_dev", "psi_last_q", "max_adj_psi", "jump_quarter",
                 "psi_fwd", "missing_jump", "psi_status", "cause_type", "psi_decision",
                 "L3_pass", "L3_reason"] + stress_cols
l3_merge = l3_df[[c for c in l3_merge_cols if c in l3_df.columns]].copy()

# Drop existing L3 columns from flog (idempotent re-run)
l3_only = [c for c in l3_merge.columns if c != "feature"]
flog_clean = flog.drop(columns=[c for c in l3_only if c in flog.columns])
flog_updated = flog_clean.merge(l3_merge, on="feature", how="left")

# Update dropped_at for L3 drops (only if not already dropped)
mask_l3_drop = (flog_updated["L3_pass"] == "N") & (flog_updated["dropped_at"].isna())
flog_updated.loc[mask_l3_drop, "dropped_at"] = "L3"
flog_updated.loc[mask_l3_drop, "drop_rule"] = flog_updated.loc[mask_l3_drop, "L3_reason"]

# Ensure no void columns
for c in flog_updated.columns:
    if flog_updated[c].isna().all():
        flog_updated[c] = flog_updated[c].astype(str)

# Save to Delta table
sdf = spark.createDataFrame(flog_updated)
sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FLOG_TBL)
print(f"Saved table: {FLOG_TBL}  ({spark.table(FLOG_TBL).count()} rows)")

# Save CSV
csv_path = os.path.join(ARTIFACTS, "feature_log.csv")
flog_updated.to_csv(csv_path, index=False)
print(f"Saved CSV : {csv_path}")

# ── Reports ─────────────────────────────────────────────────────
tmp_xl = os.path.join(_TMP, "03_psi.xlsx")

# Load policy_changes.csv for overlay
pc_path = os.path.join(PROJECT_ROOT, "config", "policy_changes.csv")
policy_changes = pd.read_csv(pc_path) if os.path.exists(pc_path) else pd.DataFrame()

with pd.ExcelWriter(tmp_xl, engine="openpyxl") as writer:
    meta = pd.DataFrame([{
        "module": "E7_L3", "run_time": datetime.now().isoformat(),
        "config_hash": common.config_hash(cfg),
        "funnel_config_hash": funnel_config_hash,
        "dev_p1_rows": len(dev_p1), "fwd_p1_rows": len(fwd_p1),
        "stress_p1_rows": len(stress_p1),
        "features_in": len(l3_df), "features_pass": int(n_pass),
        "features_drop": int(n_drop),
        "green": int(n_green), "yellow": int(n_yellow), "red": int(n_red),
    }])
    meta.to_excel(writer, sheet_name="meta", index=False)

    # Summary
    l3_df.sort_values(["psi_status", "psi_max_dev"], ascending=[True, False]).to_excel(
        writer, sheet_name="summary", index=False
    )

    # Quarterly PSI matrix
    psi_q_df.to_excel(writer, sheet_name="quarterly_psi", index=False)

    # Adjacent PSI matrix
    psi_adj_df.to_excel(writer, sheet_name="adjacent_psi", index=False)

    # Quarterly missing-rate matrix
    missing_rate_df.to_excel(writer, sheet_name="missing_rate", index=False)

    # Red-feature sheet with policy_changes overlay
    if len(red_dec_df) > 0:
        red_report = red_dec_df.copy()
        if len(policy_changes) > 0:
            red_report["policy_changes_note"] = "See policy_changes.csv"
        red_report.to_excel(writer, sheet_name="red_features", index=False)

    # psi_decisions_todo: red features without a row in psi_decisions.csv
    todo_feats = red_dec_df[red_dec_df["source"] == "no_row"][["feature", "cause_type", "decision"]] if len(red_dec_df) > 0 else pd.DataFrame()
    todo_feats.to_excel(writer, sheet_name="psi_decisions_todo", index=False)

shutil.copyfile(tmp_xl, os.path.join(REPORTS, "03_psi.xlsx"))
print(f"Saved report: {REPORTS}/03_psi.xlsx")

In [0]:
# ── Acceptance Criteria (E7 — L3) ────────────────────────────────
print("=" * 60)
print("ACCEPTANCE CRITERIA (E7 — L3)")
print("=" * 60)
checks = []

# 1. Every red feature has a decision (missing = UNKNOWN, dropped)
red_in_l3 = l3_df[l3_df["psi_status"] == "red"]
if len(red_in_l3) > 0:
    all_decided = red_in_l3["L3_pass"].notna().all()
    # All undecided reds should be dropped as UNKNOWN
    undecided = red_in_l3[red_in_l3["psi_decision"].isna() & (red_in_l3["L3_pass"] != "N")]
    checks.append((f"Every red feature has a decision ({len(undecided)} undecided)", len(undecided) == 0))
else:
    checks.append(("Every red feature has a decision (no red features)", True))

# 2. No drop is caused by STRESS
stress_drops = l3_df[
    (l3_df["L3_pass"] == "N") &
    l3_df["L3_reason"].astype(str).str.contains("STRESS", na=False)
]
checks.append((f"No drop caused by STRESS ({len(stress_drops)} stress drops)", len(stress_drops) == 0))

# 3. Baseline and comparison use the same status filter
# Verified by construction: both use P1 (PSI_STATUSES filter)
checks.append(("Baseline and comparison use same status filter (P1)", True))

# 4. Zero OOT rows read
# We never loaded OOT into any pandas DataFrame
checks.append(("Zero OOT rows read into analysis", True))

# 5. Every L2-passed feature has L3 columns
l2_passed_in_log = flog_updated[flog_updated["L2_pass"] == "Y"]
missing_l3 = l2_passed_in_log[l2_passed_in_log["L3_pass"].isna()]
checks.append((f"All L2-passed features have L3 columns ({len(missing_l3)} missing)", len(missing_l3) == 0))

# 6. feature_log.csv exported
checks.append(("feature_log.csv exported", os.path.exists(csv_path)))

# 7. Feature log table row count
tbl_ct = spark.table(FLOG_TBL).count()
checks.append((f"Feature log table rows = {tbl_ct}", tbl_ct == len(flog_updated)))

# 8. Report saved
checks.append(("03_psi.xlsx report saved", os.path.exists(os.path.join(REPORTS, "03_psi.xlsx"))))

all_pass = True
for desc, ok in checks:
    status = "\u2705 PASS" if ok else "\u274c FAIL"
    print(f"  {status} : {desc}")
    if not ok:
        all_pass = False

# ── Run summary ─────────────────────────────────────────────────
summary = {
    "module": "E7_L3",
    "run_time": datetime.now().isoformat(),
    "elapsed_s": round(time.time() - _t0, 1),
    "config_hash": common.config_hash(cfg),
    "funnel_config_hash": funnel_config_hash,
    "dev_p1_rows": len(dev_p1),
    "fwd_p1_rows": len(fwd_p1),
    "stress_p1_rows": len(stress_p1),
    "features_in": len(l3_df),
    "features_pass": int(n_pass),
    "features_drop": int(n_drop),
    "green": int(n_green), "yellow": int(n_yellow), "red": int(n_red),
    "fwd_available": fwd_bins is not None,
    "stress_available": stress_bins is not None,
    "pass_fail": "PASS" if all_pass else "FAIL",
}
os.makedirs(os.path.join(REPORTS, "run_summaries"), exist_ok=True)
with open(os.path.join(REPORTS, "run_summaries", "s03_L3.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\nRun summary: {REPORTS}/run_summaries/s03_L3.json")

for desc, ok in checks:
    assert ok, f"ACCEPTANCE FAILED: {desc}"

print("\n\u2705 All acceptance criteria passed.")